In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import re

# Model and Tokenizer Initialization
tokenizer = AutoTokenizer.from_pretrained("/home/samarth/SEM6/INLP/Project/final_model/Best_model/fine_tuned_impyadav_GPT2_hindi_lyrics")
model = AutoModelForCausalLM.from_pretrained("/home/samarth/SEM6/INLP/Project/final_model/Best_model/fine_tuned_impyadav_GPT2_hindi_lyrics")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Add special tokens if they don't already exist
special_tokens = [
    "[Sentiment]", "[Rhyme]", "[Tempo]", "[Energy]", "[Loudness]",
    "[Danceability]", "[Speechiness]", "[Acousticness]", "[Instrumentalness]",
    "[Liveness]", "[Valence]", "[Explicit]", "[Popularity]", "[Chroma]",
    "[SpectralContrast]", "[ZeroCrossings]", "[RhymeScheme]", "[Stanza]",
    "[EndStanza]", "[Line]", "[EndLine]"
]
tokens_to_add = [token for token in special_tokens if token not in tokenizer.get_vocab()]
if tokens_to_add:
    tokenizer.add_special_tokens({'additional_special_tokens': tokens_to_add})
    model.resize_token_embeddings(len(tokenizer))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Common Hindi rhyme endings
common_endings = {
    'A': 'na', 'B': 'ye', 'C': 'ta', 'D': 'aa', 'E': 'ra', 'F': 'di', 'G': 'ki',
    'H': 'se', 'I': 'le', 'J': 'ho', 'K': 'ka', 'L': 'ja', 'M': 'ni', 'N': 'me',
    'O': 'ga', 'P': 'sa'
}

def format_lyrics(generated_text, words_per_line=5, lines_per_paragraph=4):
    """
    Format the generated lyrics by removing tags, splitting into lines with a specified number of words,
    and inserting paragraph breaks after a specified number of lines.

    Args:
        generated_text (str): The raw generated text from the model.
        words_per_line (int): Number of words per line (default: 5).
        lines_per_paragraph (int): Number of lines per paragraph (default: 4).

    Returns:
        str: Formatted lyrics.
    """
    # Remove the prompt section
    prompt_end = generated_text.find("[Stanza]1[EndStanza]")
    if prompt_end != -1:
        lyrics_text = generated_text[prompt_end + len("[Stanza]1[EndStanza]"):]
    else:
        lyrics_text = generated_text

    # Remove tags like [A], [B], [C], etc.
    lyrics_text = re.sub(r'\[[A-Z]\]', '', lyrics_text)
    lyrics_text = re.sub(r'\[EndLine\]', '', lyrics_text)
    lyrics_text = re.sub(r'\[\w+\]', '', lyrics_text)  # Remove any remaining tags like [G]

    # Split the text into words
    words = lyrics_text.split()

    # Group words into lines
    lines = []
    for i in range(0, len(words), words_per_line):
        line = ' '.join(words[i:i + words_per_line])
        lines.append(line)

    # Group lines into paragraphs
    paragraphs = []
    for i in range(0, len(lines), lines_per_paragraph):
        paragraph = '\n'.join(lines[i:i + lines_per_paragraph])
        paragraphs.append(paragraph)

    # Join paragraphs with double newlines
    formatted_lyrics = '\n\n'.join(paragraphs)

    return formatted_lyrics

def generate_lyrics(
    seed_phrase="",
    sentiment="Neutral",
    rhyme_pattern="AABB",
    tempo=120,
    energy=0.5,
    loudness=-30,
    danceability=0.5,
    speechiness=0.5,
    acousticness=0.5,
    instrumentalness=0.5,
    liveness=0.5,
    valence=0.5,
    explicit="False",
    popularity=50,
    chroma=0.5,
    spectral_contrast=0.5,
    zero_crossings=0.5,
    max_new_tokens=200,
    lines_per_stanza=10
):
    """
    Generate Hindi lyrics using a fine-tuned GPT-2 model.

    Args:
        seed_phrase (str): Starting phrase for the lyrics (optional).
        sentiment (str): Desired sentiment (e.g., "Romantic", "Sad").
        rhyme_pattern (str): Rhyme scheme (e.g., "AABB", "ABAB").
        tempo (float): Musical tempo (default: 120).
        energy (float): Energy level (0-1, default: 0.5).
        loudness (float): Loudness in dB (default: -30).
        danceability (float): Danceability score (0-1, default: 0.5).
        speechiness (float): Speechiness score (0-1, default: 0.5).
        acousticness (float): Acousticness score (0-1, default: 0.5).
        instrumentalness (float): Instrumentalness score (0-1, default: 0.5).
        liveness (float): Liveness score (0-1, default: 0.5).
        valence (float): Valence score (0-1, default: 0.5).
        explicit (str): Explicit content flag ("True" or "False", default: "False").
        popularityollis(int): Popularity score (0-100, default: 50).
        chroma (float): Chroma feature (0-1, default: 0.5).
        spectral_contrast (float): Spectral contrast (0-1, default: 0.5).
        zero_crossings (float): Zero crossings rate (0-1, default: 0.5).
        max_new_tokens (int): Number of new tokens to generate (default: 200).
        lines_per_stanza (int): Number of lines per stanza (default: 4).

    Returns:
        str: Formatted lyrics with stanzas and rhyme labels.
    """
    model.eval()

    # Generate rhyme_scheme_list and rhyme_dict
    rhyme_scheme_list = list(rhyme_pattern)  # e.g., ['A', 'A', 'B', 'B']
    unique_letters = sorted(set(rhyme_pattern))
    rhyme_dict = {letter: common_endings.get(letter, 'na') for letter in unique_letters}
    rhyme_dict_str = str(rhyme_dict)
    rhyme_scheme_str = str(rhyme_scheme_list)

    # Construct prompt
    prompt = (
        f"[Sentiment]: {sentiment} {tokenizer.eos_token} "
        f"[Rhyme]: {rhyme_dict_str} {tokenizer.eos_token} "
        f"[Tempo]: {tempo} {tokenizer.eos_token} "
        f"[Energy]: {energy} {tokenizer.eos_token} "
        f"[Loudness]: {loudness} {tokenizer.eos_token} "
        f"[Danceability]: {danceability} {tokenizer.eos_token} "
        f"[Speechiness]: {speechiness} {tokenizer.eos_token} "
        f"[Acousticness]: {acousticness} {tokenizer.eos_token} "
        f"[Instrumentalness]: {instrumentalness} {tokenizer.eos_token} "
        f"[Liveness]: {liveness} {tokenizer.eos_token} "
        f"[Valence]: {valence} {tokenizer.eos_token} "
        f"[Explicit]: {explicit} {tokenizer.eos_token} "
        f"[Popularity]: {popularity} {tokenizer.eos_token} "
        f"[Chroma]: {chroma} {tokenizer.eos_token} "
        f"[SpectralContrast]: {spectral_contrast} {tokenizer.eos_token} "
        f"[ZeroCrossings]: {zero_crossings} {tokenizer.eos_token} "
        f"[RhymeScheme]: {rhyme_scheme_str}\n"
        "[Stanza]1[EndStanza]\n"
    )
    if seed_phrase:
        prompt += seed_phrase

    # Tokenize
    inputs = tokenizer(prompt, return_tensors="pt", padding=True, truncation=True).to(device)

    # Generate
    with torch.no_grad():
        generated_ids = model.generate(
            inputs.input_ids,
            attention_mask=inputs.attention_mask,
            do_sample=True,
            temperature=0.85,
            top_p=0.92,
            top_k=50,
            repetition_penalty=1.15,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            num_return_sequences=1,
            no_repeat_ngram_size=2
        )

    # Decode
    generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=False)
    # print("Generated Text:\n", generated_text)
    # Extract lines with regex
    # pattern = r'\[Line\](.*?)\[(.*?)\]\[EndLine\]'
    # matches = re.findall(pattern, generated_text, re.DOTALL)

    # # Format lyrics
    # formatted_lyrics = ""
    # stanza_count = 1
    # for i, (line, rhyme) in enumerate(matches):
    #     if i % lines_per_stanza == 0:
    #         if i > 0:
    #             formatted_lyrics += "\n"
    #         formatted_lyrics += f"Stanza_{stanza_count}:\n"
    #         stanza_count += 1
    #     formatted_lyrics += f"{line.strip()} ({rhyme})\n"

    # # Fallback if no lines are extracted
    # if not matches:
    #     lines = [line.strip() for line in generated_text.split('\n') if line.strip() and not line.startswith('[')]
    #     for i, line in enumerate(lines):
    #         if i % lines_per_stanza == 0:
    #             if i > 0:
    #                 formatted_lyrics += "\n"
    #             formatted_lyrics += f"Stanza_{stanza_count}:\n"
    #             stanza_count += 1
    #         rhyme_label = rhyme_scheme_list[i % len(rhyme_scheme_list)]
    #         formatted_lyrics += f"{line} ({rhyme_label})\n"

    # return formatted_lyrics.strip()
    return generated_text

# Example usage
if __name__ == "__main__":
    lyrics = generate_lyrics(
        seed_phrase="Tere bina",
        sentiment="Romantic",
        rhyme_pattern="AABB",
        tempo=70,
        energy=0.5,
        danceability=0.8
    )
    

    # Format the lyrics
    formatted_lyrics = format_lyrics(lyrics, words_per_line=5, lines_per_paragraph=4)

    # Print the result
    print("Formatted Lyrics:\n")
    print(formatted_lyrics)
    # print("Generated Romantic Lyrics:\n", lyrics)
    


Formatted Lyrics:

Tere bina teri tujhko joh
jaane koi baatein na 2
Mera toh sara kahin main
apni gham ki raat se

tum aayega hum jaye nahi
haan,Honthon pe yeh taali saamne
ko tasvir mein ho gayi
chalegi aur dharti ne rang

yehi kaise milenge deewana hogi
beqaraar din hontho pe dil
me hua mushkil ban jaata
doonga milega) 1 dreams ...

I'm dreaming of you guys!
, realizationist visioner is my
heart's love every day 365
laungde rehta hai sanam mera

sun le socho loon ke
jaan loge yaaron jeet se
mere sangla diya mujhe paisa
kyun zindagi intezar laate hi

maata lagte


In [24]:
import torch
import librosa
import soundfile as sf
import re
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM

# Model and Tokenizer Initialization
tokenizer = AutoTokenizer.from_pretrained("/home/samarth/SEM6/INLP/Project/final_model/Best_model/fine_tuned_impyadav_GPT2_hindi_lyrics")
model = AutoModelForCausalLM.from_pretrained("/home/samarth/SEM6/INLP/Project/final_model/Best_model/fine_tuned_impyadav_GPT2_hindi_lyrics")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Add special tokens if they don't already exist
special_tokens = [
    "[Sentiment]", "[Rhyme]", "[Tempo]", "[Energy]", "[Loudness]",
    "[Danceability]", "[Speechiness]", "[Acousticness]", "[Instrumentalness]",
    "[Liveness]", "[Valence]", "[Explicit]", "[Popularity]", "[Chroma]",
    "[SpectralContrast]", "[ZeroCrossings]", "[RhymeScheme]", "[Stanza]",
    "[EndStanza]", "[Line]", "[EndLine]"
]
tokens_to_add = [token for token in special_tokens if token not in tokenizer.get_vocab()]
if tokens_to_add:
    tokenizer.add_special_tokens({'additional_special_tokens': tokens_to_add})
    model.resize_token_embeddings(len(tokenizer))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Common Hindi rhyme endings
common_endings = {
    'A': 'na', 'B': 'ye', 'C': 'ta', 'D': 'aa', 'E': 'ra', 'F': 'di', 'G': 'ki',
    'H': 'se', 'I': 'le', 'J': 'ho', 'K': 'ka', 'L': 'ja', 'M': 'ni', 'N': 'me',
    'O': 'ga', 'P': 'sa'
}

def extract_audio_features(y, sr):
    """Extract audio features using Librosa from an audio segment."""
    try:
        features = {}
        
        # Duration
        duration = librosa.get_duration(y=y, sr=sr)
        features["duration_ms"] = duration * 1000
        
        # Tempo
        tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
        features["tempo"] = tempo
        
        # Energy
        features["energy"] = np.mean(librosa.feature.rms(y=y))
        
        # Loudness
        S = np.abs(librosa.stft(y))
        features["loudness"] = np.mean(librosa.amplitude_to_db(S, ref=np.max))
        
        # Danceability
        _, beats = librosa.beat.beat_track(y=y, sr=sr)
        if len(beats) > 0:
            beat_strength = np.mean([y[i] ** 2 for i in beats if i < len(y)])
            beat_intervals = np.diff(beats)
            regularity = 1.0 / (np.std(beat_intervals) + 1e-10)
            features["danceability"] = np.mean([beat_strength, regularity])
        else:
            features["danceability"] = 0.0
        
        # Speechiness
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
        features["speechiness"] = np.mean(np.std(mfccs, axis=1))
        
        # Acousticness
        spectral_centroid = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
        features["acousticness"] = 1.0 - (np.mean(spectral_centroid) / (sr/2))
        
        # Instrumentalness
        spectral_flatness = librosa.feature.spectral_flatness(y=y)[0]
        features["instrumentalness"] = np.mean(spectral_flatness)
        
        # Liveness
        onset_env = librosa.onset.onset_strength(y=y, sr=sr)
        features["liveness"] = np.mean(onset_env)
        
        # Valence
        spectral_contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
        features["valence"] = np.mean(spectral_contrast)
        
        # Explicit content (placeholder)
        features["explicit"] = 0.0
        
        # Popularity (placeholder)
        features["popularity"] = 50.0
        
        # Chroma
        chroma = librosa.feature.chroma_stft(y=y, sr=sr)
        features["chroma"] = np.mean(chroma)
        
        # Spectral Contrast
        spectral_contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
        features["spectral_contrast"] = np.mean(spectral_contrast)
        
        # Zero Crossings
        zero_crossings = librosa.zero_crossings(y, pad=False)
        features["zero_crossings"] = np.mean(zero_crossings)
        
        return features
    except Exception as e:
        print(f"Error extracting features: {e}")
        return {}

def generate_lyrics_from_audio(audio_file_path, sentiment="Neutral", rhyme_pattern="AABB", chunk_duration=15, max_new_tokens=50, context_tokens=100, lines_per_chunk=4):
    """
    Generate lyrics from an audio file by chunking the audio, extracting features, and generating lyrics sequentially.
    
    Args:
        audio_file_path (str): Path to the input audio file.
        sentiment (str): Desired sentiment of the lyrics (default: "Neutral").
        rhyme_pattern (str): Rhyme scheme pattern (e.g., "AABB").
        chunk_duration (int): Duration of each audio chunk in seconds (default: 15).
        max_new_tokens (int): Maximum number of new tokens to generate per chunk (default: 50).
        context_tokens (int): Number of previous tokens to include in the prompt (default: 100).
        lines_per_chunk (int): Number of lines per stanza (default: 4).
    
    Returns:
        str: Formatted lyrics with stanzas, or None if audio loading fails.
    """
    # Load audio with error handling
    try:
        y, sr = librosa.load(audio_file_path, sr=None)
    except Exception as e:
        print(f"Error loading audio with librosa: {e}")
        try:
            # Fallback to soundfile
            y, sr = sf.read(audio_file_path)
            print("Successfully loaded audio using soundfile fallback.")
        except Exception as e2:
            print(f"Fallback loading with soundfile failed: {e2}")
            return None

    # Chunk the audio
    chunk_size = int(chunk_duration * sr)
    y_chunks = [y[i:i+chunk_size] for i in range(0, len(y), chunk_size)]
    
    # Generate rhyme_dict based on rhyme_pattern
    unique_letters = sorted(set(rhyme_pattern))
    rhyme_dict = {letter: common_endings.get(letter, 'na') for letter in unique_letters}
    rhyme_dict_str = str(rhyme_dict)
    
    # Initialize running token IDs
    running_token_ids = torch.tensor([], dtype=torch.long).to(device)
    
    for k, y_chunk in enumerate(y_chunks):
        # Extract features for the current chunk
        features = extract_audio_features(y_chunk, sr)
        if not features:
            print(f"Skipping chunk {k+1} due to feature extraction failure.")
            continue
        
        # Build features string
        features_str = (
            f"[Sentiment]: {sentiment} {tokenizer.eos_token} "
            f"[Rhyme]: {rhyme_dict_str} {tokenizer.eos_token} "
            f"[Tempo]: {features['tempo']} {tokenizer.eos_token} "
            f"[Energy]: {features['energy']} {tokenizer.eos_token} "
            f"[Loudness]: {features['loudness']} {tokenizer.eos_token} "
            f"[Danceability]: {features['danceability']} {tokenizer.eos_token} "
            f"[Speechiness]: {features['speechiness']} {tokenizer.eos_token} "
            f"[Acousticness]: {features['acousticness']} {tokenizer.eos_token} "
            f"[Instrumentalness]: {features['instrumentalness']} {tokenizer.eos_token} "
            f"[Liveness]: {features['liveness']} {tokenizer.eos_token} "
            f"[Valence]: {features['valence']} {tokenizer.eos_token} "
            f"[Explicit]: {features['explicit']} {tokenizer.eos_token} "
            f"[Popularity]: {features['popularity']} {tokenizer.eos_token} "
            f"[Chroma]: {features['chroma']} {tokenizer.eos_token} "
            f"[SpectralContrast]: {features['spectral_contrast']} {tokenizer.eos_token} "
            f"[ZeroCrossings]: {features['zero_crossings']} {tokenizer.eos_token} "
        )
        
        # Tokenize features string
        features_token_ids = tokenizer.encode(features_str, return_tensors='pt').to(device)[0]
        
        if k == 0:
            # First chunk: Start with features and initial stanza
            stanza_str = f"[Stanza]1[EndStanza]\n"
            stanza_token_ids = tokenizer.encode(stanza_str, return_tensors='pt').to(device)[0]
            prompt_token_ids = torch.cat([features_token_ids, stanza_token_ids], dim=0)
        else:
            # Subsequent chunks: Use current features and last context_tokens
            last_context = running_token_ids[-context_tokens:] if len(running_token_ids) >= context_tokens else running_token_ids
            prompt_token_ids = torch.cat([features_token_ids, last_context], dim=0)
        
        # Generate lyrics
        try:
            with torch.no_grad():
                generated_ids = model.generate(
                    prompt_token_ids.unsqueeze(0),
                    max_new_tokens=max_new_tokens,
                    do_sample=True,
                    temperature=0.85,
                    top_p=0.92,
                    top_k=50,
                    repetition_penalty=1.15,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                    no_repeat_ngram_size=2
                )
            
            # Extract new tokens (excluding the prompt)
            new_token_ids = generated_ids[len(prompt_token_ids):]
            running_token_ids = torch.cat([running_token_ids, new_token_ids], dim=0)
        except Exception as e:
            print(f"Error during generation for chunk {k+1}: {e}")
            continue
    
    # Decode the generated tokens into text
    generated_text = tokenizer.decode(running_token_ids, skip_special_tokens=False)
    
    # Extract lyrics lines with rhyme tags
    pattern = r'\[Line\](.*?)\[(.*?)\]\[EndLine\]'
    matches = re.findall(pattern, generated_text, re.DOTALL)
    
    # Format lyrics into stanzas
    formatted_lyrics = ""
    for i, (line, rhyme) in enumerate(matches):
        if i % lines_per_chunk == 0:
            if i > 0:
                formatted_lyrics += "\n"
            formatted_lyrics += f"Stanza_{(i // lines_per_chunk) + 1}:\n"
        formatted_lyrics += f"{line.strip()} ({rhyme})\n"
    
    return formatted_lyrics.strip() if formatted_lyrics else "No lyrics generated due to processing errors."

# Example usage
if __name__ == "__main__":
    audio_file = "audios/calm.mp3"
    lyrics = generate_lyrics_from_audio(audio_file, sentiment="Romantic", rhyme_pattern="AABB")
    if lyrics:
        print("Generated Lyrics:\n", lyrics)
    else:
        print("Failed to generate lyrics.")

Error loading audio with librosa: Unpacker.__init__() got an unexpected keyword argument 'strict_map_key'
Successfully loaded audio using soundfile fallback.
Error extracting features: Unpacker.__init__() got an unexpected keyword argument 'strict_map_key'
Skipping chunk 1 due to feature extraction failure.
Error extracting features: Unpacker.__init__() got an unexpected keyword argument 'strict_map_key'
Skipping chunk 2 due to feature extraction failure.
Error extracting features: Unpacker.__init__() got an unexpected keyword argument 'strict_map_key'
Skipping chunk 3 due to feature extraction failure.
Error extracting features: Unpacker.__init__() got an unexpected keyword argument 'strict_map_key'
Skipping chunk 4 due to feature extraction failure.
Error extracting features: Unpacker.__init__() got an unexpected keyword argument 'strict_map_key'
Skipping chunk 5 due to feature extraction failure.
Error extracting features: Unpacker.__init__() got an unexpected keyword argument 'stri

In [31]:
pip install msgpack==1.0.5 

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [32]:
import librosa, audioread, soundfile, msgpack
print("librosa:", librosa.__version__)
print("audioread:", audioread.__version__)
print("soundfile:", soundfile.__version__)
print("msgpack:", msgpack.__version__)

librosa: 0.10.2.post1
audioread: 3.0.1
soundfile: 0.12.1


AttributeError: module 'msgpack' has no attribute '__version__'